# Wave Function Collapse

A self-contained refresher on **Wave Function Collapse (WFC)** — a constraint-solving algorithm
for procedural generation that grows a large output (a tilemap, texture, level) so that **every
local neighbourhood obeys a set of adjacency rules**, usually *learned from a single small
example*. Named after the quantum-mechanics metaphor (each cell is a superposition of all
possible tiles that "collapses" to one), it was popularised by Maxim Gumin's 2016
[`mxgmn/WaveFunctionCollapse`](https://github.com/mxgmn/WaveFunctionCollapse) repo and is now a
staple of indie game level/texture generation.

**Domain:** Procedural Generation  ·  **from study list**  ·  **runnable:** yes

## 1. What & Why

**WFC generates content that looks like a sample by enforcing local consistency everywhere.**
You give it (a) a set of tiles and (b) which tiles may sit next to which — either written by
hand or *extracted automatically from one example image* — and it fills an output grid so that
**no two adjacent cells ever violate an adjacency rule**. The result is locally
indistinguishable from the input but globally novel and non-repeating.

**The problem it solves.** Hand-authoring large, varied, *coherent* content is expensive, and
pure noise (Perlin/white) gives you smooth fields, not **discrete structured layouts** — rooms
that connect, pipes that join, towns whose roads don't dead-end into walls. WFC turns "here's a
tiny example of the style I want" into "give me a 100×100 map in that style that's internally
consistent," with **zero training** and no neural net.

**It's really a constraint-satisfaction solver.** WFC is essentially a randomized, greedy CSP
solver: cells are variables, tiles are domains, adjacency rules are constraints, and it
interleaves **collapse** (assign the most-constrained variable) with **constraint propagation**
(arc-consistency / AC-3) to shrink everyone else's options.

**Reach for it when:**
- You want output that **mimics the local texture of a small example** without training a model.
- Your content is **discrete and rule-bound**: tilemaps, dungeons, circuits, pipe networks,
  Sudoku-like layouts, poetry/text grids, 3D voxel "towns."
- You want **controllable** generation — bias weights, pre-place tiles, add hard constraints.

**Look elsewhere when:**
- You need **smooth continuous fields** (terrain height, clouds) → Perlin/Simplex noise.
- You need **global guarantees** (every room reachable, exact item counts) — WFC enforces *local*
  rules; global properties need post-processing or extra constraints, and it can **contradict**.
- Your space is huge and rules are tight — naive WFC can get slow and fail; you may want
  backtracking, hierarchical WFC, or a dedicated SAT/ASP solver.

## 2. Mental Model

**Think Sudoku, not painting.** Every cell starts as a *superposition* — it could be **any**
tile. Solving is a loop of two moves:

1. **Observe / collapse.** Pick the cell with the **lowest entropy** (fewest remaining options —
 the "most constrained" cell, like the Sudoku square with only one candidate left). Randomly
 choose one of its allowed tiles, weighted by frequency. It's now a single definite tile.
2. **Propagate.** That choice forbids some neighbours' tiles (a sea cell can't sit beside land).
 Remove those, which may further constrain *their* neighbours — ripple the consequences outward
 until everything is consistent again (this is **arc consistency / AC-3**).

Repeat until every cell is collapsed (success) or some cell's option set becomes **empty**
(a *contradiction* — back up or restart).

```
   wave (each cell = set of still-possible tiles)
   ┌─────────┬─────────┬─────────┐
   │ {L,C,S} │ {L,C,S} │ {L,C,S} │   1. all cells fully uncertain
   ├─────────┼─────────┼─────────┤
   │ {L,C,S} │  {L} ◄──┼─ collapse lowest-entropy cell to one tile
   ├─────────┼─────────┼─────────┤
   │ {L,C,S} │ {L,C,S} │ {L,C,S} │
   └─────────┴─────────┴─────────┘
              │ propagate: L forbids S next to it
              ▼
   neighbour {L,C,S} → {L,C}   (and that may ripple further)
```

The "wave function" is the grid of option-sets; "collapse" is forcing one cell to a definite
value; propagation is the wave of consequences. **Lowest-entropy-first** is the heart of the
heuristic — committing where you have the least freedom keeps contradictions rare.

## 3. Key Concepts

- **Cell / wave.** The output is a grid of cells; the **wave** is the per-cell *set of tiles
  still possible*. Generation shrinks these sets to size 1.
- **Superposition & collapse.** A cell holding multiple options is "in superposition";
  **collapsing** picks one tile (weighted random) and discards the rest.
- **Entropy.** A measure of a cell's remaining uncertainty. Crudely it's the option count;
  properly it's the **Shannon entropy of the weighted options**, `-Σ pᵢ log pᵢ`. WFC always
  collapses the **minimum-entropy** cell next (ties broken randomly, often with tiny noise).
- **Adjacency rules / constraints.** For each direction, which tiles may neighbour which. The
  two ways to get them:
  - **Simple Tiled Model** — you author the tiles and adjacencies by hand (e.g. "sea touches
    coast, never land").
  - **Overlapping Model** — extract every `N×N` patch from one example image; two patterns may
    be neighbours iff their overlap agrees. This is what makes WFC feel "magic": rules are
    *learned* from a single picture.
- **Propagation (arc consistency / AC-3).** After a collapse, repeatedly remove from each cell
  any tile that has **no compatible tile** in a neighbouring cell, queueing changed cells until
  the grid reaches a fixed point.
- **Weights.** Tiles/patterns carry frequencies (from the sample or hand-set) that bias the
  random collapse, so common patterns appear more often.
- **Contradiction.** A cell whose option set becomes **empty** — no tile fits its neighbours.
  WFC must then **backtrack** (undo recent choices) or **restart** the whole grid.
- **Symmetry / rotations & reflections.** Tilesets often auto-generate rotated/mirrored variants
  so you author fewer base tiles.
- **Determinism via seed.** Same seed + same rules ⇒ same output; reproducible and seekable.

## 4. Setup

WFC is pure logic over sets — **the Python standard library is all you need**. Everything below
runs CPU-only in milliseconds; no GPU, no network, no API key, no third-party install.

```bash
# Nothing required — we use only `random` and `collections` from the stdlib.
# For real projects you'd reach for a maintained implementation instead of hand-rolling:
%pip install wfc            # Python WFC (overlapping + tiled models)
# or use the reference C#/JS:  https://github.com/mxgmn/WaveFunctionCollapse
```

We implement a compact WFC core from scratch so the **collapse + propagate** loop is fully
visible, then drive it three ways: a hand-authored *Simple Tiled* model, a *learned* Overlapping
model, and an edge-matching pipe tileset. A single generic `wfc(...)` powers all three — the
only thing that changes is the tiles, weights, and adjacency table.

In [ ]:
import random
from collections import Counter

print("Pure stdlib — no installs needed. Python random + collections.")

# Four directions; OPP[d] is the reverse of d. Cells are addressed (x, y).
DIRS = {"N": (0, -1), "E": (1, 0), "S": (0, 1), "W": (-1, 0)}
OPP = {"N": "S", "S": "N", "E": "W", "W": "E"}


def wfc(width, height, tiles, weights, allowed, rng, max_restarts=80):
    """Generic Wave Function Collapse over a grid.

    tiles    : list of tile keys (chars, ints, anything hashable)
    weights  : {tile: weight} bias for the random collapse
    allowed  : {direction: {tile: set(tiles allowed in that direction)}}
    Returns (grid, n_restarts). Restarts the whole wave on contradiction.
    """
    for attempt in range(max_restarts):
        wave = [[set(tiles) for _ in range(width)] for _ in range(height)]
        if _solve(width, height, wave, weights, allowed, rng):
            grid = [[next(iter(wave[y][x])) for x in range(width)]
                    for y in range(height)]
            return grid, attempt
    raise RuntimeError(f"WFC failed after {max_restarts} restarts")


def _solve(width, height, wave, weights, allowed, rng):
    while True:
        # 1. OBSERVE: find the lowest-entropy (fewest-options, >1) cell.
        target, fewest = None, 1 << 30
        for y in range(height):
            for x in range(width):
                n = len(wave[y][x])
                if n == 0:
                    return False                      # contradiction
                if 1 < n < fewest:
                    fewest, target = n, (x, y)
        if target is None:
            return True                               # all cells collapsed
        # 2. COLLAPSE: pick one tile, weighted by frequency.
        x, y = target
        opts = list(wave[y][x])
        choice = rng.choices(opts, weights=[weights[t] for t in opts])[0]
        wave[y][x] = {choice}
        # 3. PROPAGATE the consequences.
        if not _propagate(width, height, wave, allowed, x, y):
            return False


def _propagate(width, height, wave, allowed, x0, y0):
    """Arc-consistency (AC-3): ripple constraints out from (x0, y0)."""
    stack = [(x0, y0)]
    while stack:
        x, y = stack.pop()
        for d, (dx, dy) in DIRS.items():
            nx, ny = x + dx, y + dy
            if not (0 <= nx < width and 0 <= ny < height):
                continue
            # Tiles the current cell can support in direction d:
            support = set()
            for t in wave[y][x]:
                support |= allowed[d][t]
            before = len(wave[ny][nx])
            wave[ny][nx] &= support                   # drop unsupported tiles
            if not wave[ny][nx]:
                return False                          # neighbour wiped out
            if len(wave[ny][nx]) < before:
                stack.append((nx, ny))                # changed → re-check it
    return True


print("Core ready: wfc(), _solve(), _propagate()")

## 5. Worked Examples

### Example 1 — Simple Tiled Model: a land / coast / sea map

The classic introductory tileset. Three tiles — **L**and, **C**oast, **S**ea — with one rule:
*sea may never touch land directly; coast must separate them.* We feed those adjacencies (plus
frequency weights) to the core and out comes a coherent coastline. Then we **verify** the
invariant held: zero land–sea adjacencies anywhere.

In [ ]:
def build_adjacency(pairs, tiles):
    """Turn a set of undirected, isotropic 'a may touch b' pairs into the
    directional `allowed` table the core expects (same rule in all 4 dirs)."""
    allowed = {d: {t: set() for t in tiles} for d in DIRS}
    undirected = set()
    for a, b in pairs:
        undirected |= {(a, b), (b, a)}
    for a, b in undirected:
        for d in DIRS:
            allowed[d][a].add(b)
    return allowed


tiles = ["L", "C", "S"]
weights = {"L": 4, "C": 2, "S": 3}
pairs = [("L", "L"), ("L", "C"), ("C", "C"), ("C", "S"), ("S", "S")]  # no L–S!
allowed = build_adjacency(pairs, tiles)

grid, restarts = wfc(28, 12, tiles, weights, allowed, random.Random(42))

glyph = {"L": "#", "C": ".", "S": " "}
print(f"generated 28x12 map (restarts: {restarts})\n")
for row in grid:
    print("".join(glyph[t] for t in row))

# Verify the adjacency constraint actually held everywhere.
violations = 0
for y in range(12):
    for x in range(28):
        for d, (dx, dy) in DIRS.items():
            nx, ny = x + dx, y + dy
            if 0 <= nx < 28 and 0 <= ny < 12:
                if {grid[y][x], grid[ny][nx]} == {"L", "S"}:
                    violations += 1
print(f"\nland-sea adjacencies (must be 0): {violations}")

### Example 2 — Overlapping Model: *learn* the rules from one tiny image

This is the famous WFC mode. Instead of authoring rules, we hand it a **5×5 sample** of a pond
surrounded by coast, surrounded by land, and let it **extract every `N×N` pattern** and infer
adjacency: pattern *A* may sit to the *east* of *B* iff their overlapping region agrees. We then
run the **same core** over the patterns and render a larger map. Note how the output invents new
arrangements of ponds while perfectly preserving the *local* "pond ⊂ coast ⊂ land" structure —
no rule was written by hand.

In [ ]:
def patterns_from_sample(sample, N):
    """Extract every NxN pattern (toroidal wrap) and its frequency."""
    h, w = len(sample), len(sample[0])
    pats = Counter()
    for y in range(h):
        for x in range(w):
            pat = tuple(tuple(sample[(y + dy) % h][(x + dx) % w]
                              for dx in range(N)) for dy in range(N))
            pats[pat] += 1
    return pats


def overlap_ok(p1, p2, dx, dy, N):
    """True if pattern p2, offset by (dx, dy), agrees with p1 where they overlap."""
    for y in range(N):
        for x in range(N):
            x2, y2 = x - dx, y - dy
            if 0 <= x2 < N and 0 <= y2 < N and p1[y][x] != p2[y2][x2]:
                return False
    return True


def overlapping_rules(pats, N):
    """Build (pattern list, weights, allowed) from extracted patterns."""
    plist = list(pats)
    allowed = {d: {i: set() for i in range(len(plist))} for d in DIRS}
    for i, p1 in enumerate(plist):
        for j, p2 in enumerate(plist):
            for d, (dx, dy) in DIRS.items():
                if overlap_ok(p1, p2, dx, dy, N):
                    allowed[d][i].add(j)
    weights = {i: pats[p] for i, p in enumerate(plist)}
    return plist, weights, allowed


sample = [
    "LLLLL",
    "LCCCL",
    "LC.CL",   # '.' = pond water, surrounded by coast 'C', surrounded by land 'L'
    "LCCCL",
    "LLLLL",
]
N = 3
pats = patterns_from_sample(sample, N)
plist, weights, allowed = overlapping_rules(pats, N)
print(f"learned {len(plist)} distinct {N}x{N} patterns from the {len(sample)}x"
      f"{len(sample[0])} sample\n")

idx_grid, restarts = wfc(20, 12, list(range(len(plist))), weights, allowed,
                         random.Random(3))
# Each collapsed cell holds a pattern index; render its top-left glyph.
print(f"generated 20x12 map (restarts: {restarts})\n")
for row in idx_grid:
    print("".join(plist[i][0][0] for i in row))

### Example 3 — Edge-matching pipes (a richer tileset) + restart counts

Adjacency doesn't have to be "which tile types touch" — it can be **connector matching**. Here
each pipe tile declares which of its four edges carries a connector; two tiles may be neighbours
only if the touching edges **agree** (both connected or both empty). The core then weaves a fully
connected pipe network where every line joins up. We also run several seeds and report how often
the solver had to **restart on a contradiction** — for this permissive ruleset, rarely.

In [ ]:
# Each tile: (N, E, S, W) edge has a connector (1) or not (0).
PIPES = {
    " ": (0, 0, 0, 0),
    "─": (0, 1, 0, 1), "│": (1, 0, 1, 0),
    "┌": (0, 1, 1, 0), "┐": (0, 0, 1, 1),
    "└": (1, 1, 0, 0), "┘": (1, 0, 0, 1),
    "├": (1, 1, 1, 0), "┤": (1, 0, 1, 1),
    "┬": (0, 1, 1, 1), "┴": (1, 1, 0, 1), "┼": (1, 1, 1, 1),
}
EDGE = {"N": 0, "E": 1, "S": 2, "W": 3}


def pipe_rules():
    tiles = list(PIPES)
    allowed = {d: {t: set() for t in tiles} for d in DIRS}
    for a in tiles:
        for b in tiles:
            for d in DIRS:
                # b sits in direction d of a: a's d-edge must match b's opposite edge.
                if PIPES[a][EDGE[d]] == PIPES[b][EDGE[OPP[d]]]:
                    allowed[d][a].add(b)
    weights = {t: (1 if t == " " else 2) for t in tiles}
    return tiles, weights, allowed


tiles, weights, allowed = pipe_rules()
restart_counts = [wfc(24, 10, tiles, weights, allowed, random.Random(s))[1]
                  for s in range(8)]
print("restarts across 8 seeds:", restart_counts, "\n")

grid, _ = wfc(24, 10, tiles, weights, allowed, random.Random(0))
for row in grid:
    print("".join(row))

## 6. Gotchas & Pitfalls

- **Contradictions are inevitable on tight rulesets.** When a cell's options hit empty, you're
  stuck. The cheap fix (used here) is **restart the whole grid**; the proper fix is
  **backtracking** (undo the last few collapses) or storing checkpoints. Naive restart can loop
  forever on over-constrained inputs — always cap retries.
- **It's a *local* solver — no global guarantees.** WFC ensures every adjacency is legal, but
  *not* that "the dungeon is fully connected" or "there's exactly one exit." Global properties
  need extra constraints, pre-placed tiles, or a post-processing/repair pass.
- **Min-entropy choice matters.** Always collapse the **most-constrained** cell, not a random or
  scan-order cell — picking the freest cell first massively increases contradictions. Real
  implementations use **Shannon entropy with weights** (+ tiny tie-break noise), not just the
  raw option count we use for clarity here.
- **Propagation must reach a fixed point.** If you only update *immediate* neighbours and stop,
  you'll leave illegal states latent and hit contradictions later. Keep a worklist and ripple
  until nothing changes (AC-3), as `_propagate` does.
- **Overlapping-model blowup.** Pattern count grows fast with `N` and with rotations/reflections;
  adjacency is `O(patterns²)` to build. Keep `N` small (2–3) and the sample tiny, or precompute
  adjacency once and cache it.
- **Edge/boundary handling.** Toroidal (wrap) vs. fixed borders changes results and which
  patterns are legal at the edges. Decide deliberately; mismatches here cause surprise
  contradictions in corners.
- **Weights bias, they don't constrain.** A weight of 0 isn't the same as a forbidden adjacency.
  Use the **adjacency table** for "never," weights only for "how often."
- **Determinism needs a seeded RNG.** Pass your own `random.Random(seed)`; relying on the global
  `random` makes runs irreproducible and hard to debug.
- **Performance.** Pure-Python WFC is fine for tens-of-thousands of cells; for big maps use
  bitmask domains (ints, not Python sets), precomputed support counts, and a real backtracker —
  or a maintained library / the reference C# implementation.

## 7. When to Use vs Alternatives

| Approach | Best for | Trade-off vs WFC |
|---|---|---|
| **Wave Function Collapse** | Locally-coherent *discrete* output learned from a small example: tilemaps, textures, pipes, decorated rooms. | Only *local* guarantees; can contradict; can be slow on big/tight problems. |
| **Perlin / Simplex noise** | Smooth *continuous* fields: terrain height, clouds, fog. | No discrete tiles or adjacency rules — different problem entirely. See `perlin-noise`. |
| **Cellular automata** | Organic blobby caves, cave smoothing, growth. | No notion of "matches this example"; rules are evolution steps, not adjacency constraints. See `cellular-automata`. |
| **BSP / room-and-corridor** | Dungeons with *guaranteed* connectivity and room structure. | Strong global structure but a recognisable, less organic look. See `bsp-dungeon-generation`. |
| **Markov chains / n-grams** | 1-D sequences (text, music) "in the style of" a sample. | WFC generalises the same idea to 2-D/3-D grids with hard constraints. See `markov-chains-ngrams`. |
| **L-systems / grammars** | Recursive, branching structure: plants, roads, buildings. | Rule-rewriting, not neighbour-matching; great for hierarchy, weak for free-form texture. See `l-systems`. |
| **General SAT / CSP / ASP solver** | When you need *completeness* and complex global constraints. | Heavier, less "creative-random"; WFC is a fast randomized greedy CSP tuned for variety. |
| **GAN / diffusion PCG** | Photoreal or highly varied content from large datasets. | Needs training data + compute; WFC needs *one* example and no training. See `gan-diffusion-pcg`. |

**Rule of thumb:** if your content is **discrete, tile-like, and you can show it one good
example**, WFC is the sweet spot — cheap, training-free, controllable. If you need **smooth
fields**, reach for noise; if you need **guaranteed global structure**, reach for BSP or a real
CSP solver (or run WFC and *repair* the result).

## 8. Resources

- **Maxim Gumin — `mxgmn/WaveFunctionCollapse` (the original repo + GIFs that started it all)** —
  https://github.com/mxgmn/WaveFunctionCollapse
- **Robert Heaton — "The Wavefunction Collapse Algorithm explained very clearly"** —
  https://robertheaton.com/2018/12/17/wavefunction-collapse-algorithm/
- **BorisTheBrave — "WFC tips and tricks" + "The Algorithm" (deep, practical writeups)** —
  https://www.boristhebrave.com/2020/04/13/wave-function-collapse-explained/
- **Isaac Karth & Adam Smith — "WaveFunctionCollapse is Constraint Solving in the Wild" (PCG 2017 paper)** —
  https://adamsmith.as/papers/wfc_is_constraint_solving_in_the_wild.pdf
- **Marian Kleineberg — "Infinite procedurally generated city" (3D WFC, video + writeup)** —
  https://marian42.de/article/wfc/
- **`fast-wfc` (high-performance C++ implementation) and Python ports** —
  https://github.com/math-fehr/fast-wfc
- Related notebooks in this domain: `cellular-automata`, `bsp-dungeon-generation`,
  `markov-chains-ngrams`, `l-systems`, `perlin-noise`.

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
def propagate(domains, neighbours, allowed, seeds):
    """Ripple the consequences of a decision outward until the grid is consistent."""
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE